# Push-up form classifier - Colab training runner

The Kaggle dataset `mohamadashrafsalama/pushup` is **raw videos** (Correct/Wrong sequence/*.mp4),
not a feature CSV. So the TRAIN cell runs **MediaPipe pose extraction per video**, segments reps,
and builds rep-level features (same definitions as the app's `:core` PushUpFeatureExtractor) before
training. Extraction of ~100 videos takes a **few minutes**.

Run top to bottom. Mount Drive with the **kimgt2828** account; complete the Kaggle login form.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/health_training'
RUNS_DIR   = f'{DRIVE_ROOT}/runs'
print(DRIVE_ROOT)

In [ ]:
# Clone the model-training branch (has the push-up video pipeline). %cd /content first = re-run safe.
%cd /content
!rm -rf /content/health_trainer
!git clone --branch model-training --single-branch https://github.com/kimgt0128/health-trainer.git /content/health_trainer
%cd /content/health_trainer
# Push-up training extracts landmarks from video -> needs mediapipe + opencv (+ kagglehub).
!pip install -q mediapipe opencv-python "kagglehub[pandas-datasets]"

In [ ]:
# Kaggle auth - enter username + API token, wait for the green confirmation, then continue.
import kagglehub
kagglehub.login()

In [ ]:
# === TRAIN === downloads the dataset + the MediaPipe .task, extracts per-video landmarks, segments
# reps, builds rep-level features, trains HGB, writes 4 artifacts. ~100 videos -> a few minutes.
%cd /content/health_trainer
import os
os.environ['PYTHONPATH'] = '/content/health_trainer/ml/src'
!python ml/src/train_pushup_form_classifier.py \
    --run-dir "$RUNS_DIR/pushup_form_classifier_v1" \
    --test-size 0.2 \
    --random-state 42

In [ ]:
# Inspect artifacts written to Drive (4 files expected).
!find "$RUNS_DIR/pushup_form_classifier_v1" -maxdepth 1 -type f -print
!cat "$RUNS_DIR/pushup_form_classifier_v1/metrics_summary.json"

## Export path: Keras -> TFLite (the artifact the app loads)

The sklearn model above (`pushup_form_classifier.joblib`) does **not** convert to TFLite, so the
app can't load it. The cell below runs `train_pushup_form_mlp.py` instead: it trains a **tiny,
regularized Keras model** (default = logistic-regression head, the safe capacity for the ~54-rep
dataset) with `Normalization` baked **inside** the model (so the app feeds RAW features), reuses
the SAME rep-level rows + group-by-clip split + grouped CV, and writes the 4 artifacts the app
expects — including `pushup_form.tflite`.

Honest note: this is a **weak baseline assist** (rules stay primary; the app suppresses
low-confidence / `correct` predictions). The durable deliverable is this export PATH — it
produces a better model as more (self-filmed, rep-labelled) data arrives. The CV has high
variance (~0.835 +/- 0.133) because n is tiny — do not overstate it.

> Prerequisite: the clone cell above must fetch a branch that **contains** `ml/src/train_pushup_form_mlp.py` (this script). Point the `git clone --branch` cell at the branch where this trainer has landed (e.g. once merged to `model-training`).

In [ ]:
# === EXPORT (Keras -> TFLite) === same data pipeline as the sklearn cell, but a tiny Keras
# model we CAN convert to TFLite. tensorflow is heavy + Colab-only, so install it here (the
# earlier cell only installed mediapipe/opencv/kagglehub for the sklearn path).
%cd /content/health_trainer
!pip install -q tensorflow==2.17.0
import os
os.environ['PYTHONPATH'] = '/content/health_trainer/ml/src'
# Default --hidden "" = logistic-regression head (overfit-resistant on ~54 reps). Group-by-clip
# split + grouped CV are ON by default (the honest, leakage-free methodology).
!python ml/src/train_pushup_form_mlp.py \
    --run-dir "$RUNS_DIR/pushup_form_mlp_v1" \
    --test-size 0.2 \
    --random-state 42 \
    --cv-folds 5

In [ ]:
# Confirm the 4-artifact SET (the app needs all four together, not just the .tflite).
import os, json
RUN = f"{RUNS_DIR}/pushup_form_mlp_v1"
expected = ["pushup_form.tflite", "labels_pushup_form.json", "feature_config.json", "metrics_summary.json"]
present = {f: os.path.exists(os.path.join(RUN, f)) for f in expected}
print("artifacts:", json.dumps(present, indent=2))
assert all(present.values()), f"MISSING artifacts: {[f for f, ok in present.items() if not ok]}"
print("\nfeature_config.json (order is the train<->app contract):")
print(open(os.path.join(RUN, "feature_config.json")).read())
print("metrics_summary.json:")
print(open(os.path.join(RUN, "metrics_summary.json")).read())

In [ ]:
# Drop the .tflite into the app. The app :core FormClassifierRegistry PUSH_UP Spec loads
# assets/models/pushup_form.tflite with labels [correct, incorrect]. labels_pushup_form.json /
# feature_config.json travel alongside as the contract record (feature ORDER must match
# :core PushUpFeatureExtractor.FEATURE_NAMES). The app works rules-only if the model is absent.
import shutil, os
SRC = f"{RUNS_DIR}/pushup_form_mlp_v1/pushup_form.tflite"
# In Colab the app repo isn't checked out; copy the model back to Drive and move it into the
# app on your dev machine at: app/src/main/assets/models/pushup_form.tflite
DEST_DRIVE = f"{DRIVE_ROOT}/exports/pushup_form.tflite"
os.makedirs(os.path.dirname(DEST_DRIVE), exist_ok=True)
shutil.copy(SRC, DEST_DRIVE)
print("copied ->", DEST_DRIVE)
print("then on your dev machine: cp <drive>/exports/pushup_form.tflite app/src/main/assets/models/pushup_form.tflite")